In [4]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

class KMeansMajorityClassifier:
    """
    Classifier:
      - fit(): run k-means on X, then assign each cluster a label = majority y in that cluster
      - predict(): assign each new x to nearest centroid, output that cluster's label
      - score(): accuracy on (X, y)

    Notes:
      - Works with pandas DataFrames
      - One-hot encodes any non-numeric columns (e.g., CITY), and aligns columns at test time
      - Median-imputes missing values (though your data should have none) and standardizes features
    """
    def __init__(self, n_clusters=10, random_state=0, n_init="auto"):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.n_init = n_init

    def _prepare_X(self, X, fit=False):
        # Accept DataFrame or array-like
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()
        else:
            X_df = pd.DataFrame(X)

        # One-hot encode any non-numeric columns
        cat_cols = [c for c in X_df.columns if not pd.api.types.is_numeric_dtype(X_df[c])]
        X_oh = pd.get_dummies(X_df, columns=cat_cols, drop_first=False)

        if fit:
            self.feature_columns_ = X_oh.columns
        else:
            # Align test columns to training columns
            X_oh = X_oh.reindex(columns=self.feature_columns_, fill_value=0)

        # Impute + scale
        if fit:
            self.imputer_ = SimpleImputer(strategy="median")
            X_imp = self.imputer_.fit_transform(X_oh)
            self.scaler_ = StandardScaler()
            X_scaled = self.scaler_.fit_transform(X_imp)
        else:
            X_imp = self.imputer_.transform(X_oh)
            X_scaled = self.scaler_.transform(X_imp)

        return X_scaled

    def fit(self, X, y):
        y_arr = np.asarray(y).astype(int)
        X_scaled = self._prepare_X(X, fit=True)

        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=self.n_init
        )
        self.kmeans_.fit(X_scaled)

        clusters = self.kmeans_.labels_
        self.cluster_label_ = {}

        for k in range(self.n_clusters):
            idx = np.where(clusters == k)[0]
            if len(idx) == 0:
                self.cluster_label_[k] = 0
            else:
                labels = y_arr[idx]
                # Majority vote; tie breaks to 1 if mean>=0.5
                self.cluster_label_[k] = int(labels.mean() >= 0.5)

        return self

    def predict(self, X):
        X_scaled = self._prepare_X(X, fit=False)
        cluster_ids = self.kmeans_.predict(X_scaled)
        return np.array([self.cluster_label_.get(c, 0) for c in cluster_ids], dtype=int)

    def score(self, X, y):
        y_arr = np.asarray(y).astype(int)
        return accuracy_score(y_arr, self.predict(X))


In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

paths = [
    "/content/Beijing_labeled.csv",
    "/content/Shenyang_labeled.csv",
    "/content/Guangzhou_labeled.csv",
    "/content/Shanghai_labeled.csv",
    "/content/Chengdu_labeled.csv"
]

dfs = []
for p in paths:
    df = pd.read_csv(p)
    df["CITY"] = os.path.basename(p).split("_")[0]
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

# Train and validate on Beijing & Shenyang
train_cities = ["Beijing", "Shenyang"]
# Evaluate on Guangzhou and Shanghai
test_cities = ["Guangzhou", "Shanghai"]

trainval_df = data[data["CITY"].isin(train_cities)]
test_df  = data[data["CITY"].isin(test_cities)]
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.2,
    random_state=42,
    stratify=trainval_df["PM_HIGH"]
)

X_train = train_df.drop(columns=["PM_HIGH", "CITY"])
y_train = train_df["PM_HIGH"]

X_val = val_df.drop(columns=["PM_HIGH", "CITY"])
y_val = val_df["PM_HIGH"]

X_test = test_df.drop(columns=["PM_HIGH", "CITY"])
y_test = test_df["PM_HIGH"]


ks = [5, 10, 20, 30, 40, 50]
val_scores = {}

for k in ks:
    clf = KMeansMajorityClassifier(n_clusters=k, random_state=42)
    clf.fit(X_train, y_train)
    val_scores[k] = clf.score(X_val, y_val)

print("Validation accuracy by k:")
for k, acc in val_scores.items():
    print(f"k={k:2d} → {acc:.3f}")

best_k = max(val_scores, key=val_scores.get)
print("Best k:", best_k)

X_trainval = trainval_df.drop(columns=["PM_HIGH", "CITY"])
y_trainval = trainval_df["PM_HIGH"]

clf = KMeansMajorityClassifier(n_clusters=best_k, random_state=42)
clf.fit(X_trainval, y_trainval)

y_pred = clf.predict(X_test)

print("Final test accuracy:", accuracy_score(y_test, y_pred))

print("Train accuracy:", clf.score(X_train, y_train))
print("Test accuracy:", clf.score(X_test, y_test))

# Predict once
y_pred = clf.predict(X_test)

# Overall accuracy
overall_acc = accuracy_score(y_test, y_pred)
print("Overall test accuracy:", overall_acc)

# Per-city accuracy (use test_df for CITY)
results = pd.DataFrame({
    "CITY": test_df["CITY"].values,
    "y_true": y_test.values,
    "y_pred": y_pred
})

per_city_acc = results.groupby("CITY").apply(
    lambda g: accuracy_score(g["y_true"], g["y_pred"])
)

print("Per-city test accuracy:")
print(per_city_acc)

print("Train accuracy:", clf.score(X_train, y_train))
print("Trainval accuracy:", clf.score(X_trainval, y_trainval))

Validation accuracy by k:
k= 5 → 0.725
k=10 → 0.777
k=20 → 0.781
k=30 → 0.788
k=40 → 0.793
k=50 → 0.796
Best k: 50
Final test accuracy: 0.7306696263411024
Train accuracy: 0.7737478411053541
Test accuracy: 0.7306696263411024
Overall test accuracy: 0.7306696263411024
Per-city test accuracy:
CITY
Guangzhou    0.781065
Shanghai     0.680237
dtype: float64
Train accuracy: 0.7737478411053541
Trainval accuracy: 0.7789291882556131


/tmp/ipython-input-925352746.py:89: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_city_acc = results.groupby("CITY").apply(


In [6]:
from sklearn.metrics import balanced_accuracy_score

bal_acc = balanced_accuracy_score(y_test, y_pred)
print("Balanced test accuracy:", bal_acc)

Balanced test accuracy: 0.5141260229854632


In [7]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

[[1919  565]
 [ 163   56]]
